# 100-Protein Dataset Builder

Downloads AlphaFoldDB structures, extracts 3Di sequences via Foldseek, \
and saves `test_set_AA.fasta` + `test_set_3Di.fasta` to Google Drive.

**Run this notebook once before `prostT5_baseline_performance.ipynb`.**
After running, the FASTAs will be at `/content/drive/MyDrive/models/` on Colab.

In [ ]:
%pip install -q sentencepiece


In [ ]:
import os, json, gc, subprocess, shutil, stat, tarfile, platform, urllib.request
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MODELS = Path('/content/drive/MyDrive/models')
    USE_DRIVE = True
    print(f'Drive mounted. Models dir: {DRIVE_MODELS}')
except (ImportError, ModuleNotFoundError):
    print('Not on Colab — outputs saved locally only.')
    DRIVE_MODELS = None
    USE_DRIVE = False

WORK_DIR = Path('/content/dataset') if USE_DRIVE else Path('./dataset')
WORK_DIR.mkdir(parents=True, exist_ok=True)
PDB_DIR = WORK_DIR / 'pdbs'
PDB_DIR.mkdir(exist_ok=True)
FOLDSEEK_DIR = WORK_DIR / 'foldseek_bin'
print(f'Work dir: {WORK_DIR}')


In [ ]:
# 100 benchmark proteins (stratified by length, taxonomy, fold class)
TEST_IDS = [
    'P62990', 'P01308', 'P02798', 'P62944', 'P01542',
    'P43408', 'P0AFL6', 'P0ACJ8', 'P0A988', 'P0A7Y4',
    'Q57733', 'P22579', 'P15378', 'P34690', 'P0A7U3',
    'P63165', 'P0A7N4', 'P24311', 'P56252', 'P18253',
    'A0A6G0XC32', 'P42212', 'P61626', 'P68871', 'P37840',
    'P02686', 'P61769', 'P00720', 'P0ABH7', 'P0A9Q7',
    'P0A953', 'P00698', 'P0DP27', 'P62937', 'P10599',
    'P56201', 'P16949', 'P68082', 'P63101', 'Q9R0Q7',
    'P04637', 'P0DTC9', 'P0AEX9', 'P00918', 'P0A855',
    'P00784', 'P69441', 'P02572', 'P63244', 'P00439',
    'P62593', 'P00766', 'P0A6P9', 'P04406', 'P24941',
    'P02994', 'P61583', 'P05089', 'P07688', 'P06280',
    'P19367', 'P0A6F5', 'P02768', 'P08684', 'P35354',
    'P04150', 'P0AG67', 'P15056', 'P05067', 'P0ABB0',
    'P23246', 'P21802', 'P11413', 'P07901', 'Q02880',
    'P14598', 'P10591', 'P61869', 'P10275', 'P00832',
    'P38398', 'P02452', 'P12821', 'P00968', 'P42336',
    'P42345', 'P0DTC2', 'P13569', 'O60885', 'P06756',
    'P04626', 'Q13936', 'P02458', 'Q61410', 'P13368',
    'P04808', 'Q04721', 'P07949', 'P35498', 'P00533',
]
BACKUP_IDS = [
    'P0A7L0', 'P01375', 'P62988', 'P00441', 'P00517',
    'P04040', 'P22303', 'P11362', 'Q9NZC2', 'P54289',
]
print(f'{len(TEST_IDS)} main proteins + {len(BACKUP_IDS)} backups')


In [ ]:
def _ensure_foldseek():
    existing = shutil.which('foldseek')
    if existing:
        return existing
    local = FOLDSEEK_DIR / 'bin' / 'foldseek'
    if local.exists() and os.access(local, os.X_OK):
        return str(local)

    os_map = {
        'Linux':  'foldseek-linux-avx2.tar.gz',
        'Darwin': 'foldseek-osx-universal.tar.gz',
    }
    fname = os_map.get(platform.system())
    if not fname:
        raise RuntimeError(
            f'No auto-install for {platform.system()}. '
            'Install Foldseek manually or pre-upload FASTA files to Drive.'
        )

    FOLDSEEK_DIR.mkdir(parents=True, exist_ok=True)
    tarball = FOLDSEEK_DIR / 'foldseek.tar.gz'
    if not tarball.exists():
        print('Downloading Foldseek...')
        urllib.request.urlretrieve(f'https://mmseqs.com/foldseek/{fname}', tarball)
    print('Extracting...')
    with tarfile.open(tarball, 'r:gz') as tf:
        tf.extractall(FOLDSEEK_DIR)
    src = FOLDSEEK_DIR / 'foldseek' / 'bin' / 'foldseek'
    dst = FOLDSEEK_DIR / 'bin' / 'foldseek'
    dst.parent.mkdir(exist_ok=True)
    shutil.copy(src, dst)
    dst.chmod(dst.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    print(f'Foldseek installed: {dst}')
    return str(dst)


FOLDSEEK = _ensure_foldseek()
print(f'Foldseek binary: {FOLDSEEK}')


In [ ]:
def _download_pdb(uid):
    pdb = PDB_DIR / f'AF-{uid}-F1-model.pdb'
    if pdb.exists() and pdb.stat().st_size > 0:
        return pdb
    api = f'https://alphafold.ebi.ac.uk/api/prediction/{uid}'
    with urllib.request.urlopen(api, timeout=15) as r:
        meta = json.loads(r.read())
    if not meta:
        raise ValueError(f'No AlphaFoldDB entry for {uid}')
    urllib.request.urlretrieve(meta[0]['pdbUrl'], pdb)
    return pdb


def _extract_sequences(uid, pdb_file):
    work = WORK_DIR / 'foldseek_work' / uid
    work.mkdir(parents=True, exist_ok=True)
    pdb_dir = work / 'pdbs'
    pdb_dir.mkdir(exist_ok=True)
    tgt = pdb_dir / pdb_file.name
    if not tgt.exists():
        shutil.copy(pdb_file, tgt)

    db = str(work / 'db')

    def run(*args):
        subprocess.run(list(args), check=True, capture_output=True)

    run(FOLDSEEK, 'createdb', str(pdb_dir), db)
    run(FOLDSEEK, 'lndb', db + '_h', db + '_ss_h')

    def read_fasta(stem, lower=False):
        fa = work / (stem + '.fasta')
        run(FOLDSEEK, 'convert2fasta', str(work / stem), str(fa))
        seq = ''.join(line.strip() for line in fa.read_text().splitlines()
                      if not line.startswith('>'))
        return seq.lower() if lower else seq.upper()

    aa = read_fasta('db',    lower=False)
    di = read_fasta('db_ss', lower=True)
    return aa, di


print('Download + extraction helpers defined.')


In [ ]:
dataset = {}
failed = []

for i, uid in enumerate(TEST_IDS):
    try:
        pdb = _download_pdb(uid)
        aa, di = _extract_sequences(uid, pdb)
        if not aa or not di or len(aa) != len(di):
            raise ValueError(f'Length mismatch: AA={len(aa)}, 3Di={len(di)}')
        dataset[uid] = {'aa': aa, '3di': di}
        if (i + 1) % 10 == 0 or i == 0:
            print(f'[{i+1}/{len(TEST_IDS)}] OK  {uid}  L={len(aa)}')
    except Exception as e:
        print(f'FAIL  {uid}: {e}')
        failed.append(uid)

for backup_uid in BACKUP_IDS:
    if not failed:
        break
    try:
        pdb = _download_pdb(backup_uid)
        aa, di = _extract_sequences(backup_uid, pdb)
        if aa and di and len(aa) == len(di):
            dataset[backup_uid] = {'aa': aa, '3di': di}
            failed.pop(0)
            print(f'BACKUP OK  {backup_uid}  L={len(aa)}')
    except Exception as e:
        print(f'BACKUP FAIL  {backup_uid}: {e}')

print(f'\nDataset ready: {len(dataset)} proteins, {len(failed)} unresolved failures')
if failed:
    print(f'Unresolved: {failed}')


In [ ]:
AA_FASTA = WORK_DIR / 'test_set_AA.fasta'
DI_FASTA = WORK_DIR / 'test_set_3Di.fasta'

with open(AA_FASTA, 'w') as f:
    for uid, d in dataset.items():
        print('>' + uid, file=f)
        print(d['aa'], file=f)

with open(DI_FASTA, 'w') as f:
    for uid, d in dataset.items():
        print('>' + uid, file=f)
        print(d['3di'], file=f)

print(f'Saved {AA_FASTA.name}  ({len(dataset)} proteins)')
print(f'Saved {DI_FASTA.name}  ({len(dataset)} proteins)')

if USE_DRIVE:
    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    shutil.copy(AA_FASTA, DRIVE_MODELS / 'test_set_AA.fasta')
    shutil.copy(DI_FASTA, DRIVE_MODELS / 'test_set_3Di.fasta')
    print(f'\nUploaded to Drive: {DRIVE_MODELS}/')
else:
    print('\nNot on Colab — upload FASTAs manually to Google Drive at /MyDrive/models/')
